# SVM RBF: evaluasi full-training

Akram — `svm_rbf`. Parameter hasil screening sebelumnya: `{'C': 1.0, 'gamma': 'scale'}`.
Setiap model memakai seluruh train fold, tanpa subsampling: stratified 5-fold,
seed 42, feature engineering, tol 0.01, max_iter 1000. Kalibrasi hanya memakai
data train fold. Tidak ada pengiriman Kaggle atau push Git otomatis.

Run berjalan lama di CPU. Log dan checkpoint OOF per fold berada di
`output/experiments/svm_rbf/`; hasil lengkap tersimpan setelah notebook selesai.


In [1]:
import hashlib
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'models').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from models import SVM
from preprocessing import SVMPreprocessor, TARGET, clean_train
from utils import (best_threshold, confusion_matrix, cross_val_proba,
                   f1_per_class, fold_scores, macro_f1, stratified_kfold)

SEED = 42
N_FOLDS = 5
SCREEN_SIZE = 3000
KERNEL = 'rbf'
EXP_NAME = 'svm_rbf'
BASE_PARAMS = dict(kernel=KERNEL, tol=0.01, max_iter=1000, seed=SEED)
FIG_PATH = ROOT / 'output' / 'figures' / f'{EXP_NAME}.png'
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('Python', sys.version.split()[0], '| NumPy', np.__version__,
      '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

RUN_DIR = ROOT / 'output' / 'experiments' / EXP_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)


Python 3.12.6 | NumPy 2.2.6 | pandas 2.3.0 | scikit-learn 1.9.0


In [2]:
train_raw = pd.read_csv(ROOT / 'data' / 'train.csv', dtype={'id': str})
train = clean_train(train_raw)
y = train[TARGET].to_numpy()
folds = stratified_kfold(y, n_splits=N_FOLDS, seed=SEED)
assert set(np.unique(y)) == {0, 1}
assert np.array_equal(np.sort(np.concatenate([va for _, va in folds])), np.arange(len(y)))
assert all(not np.intersect1d(tr, va).size for tr, va in folds)
print('Full training:', len(train), '| fold train sizes:', [len(tr) for tr, _ in folds], flush=True)


def report(name, labels, proba, splits, seconds):
    assert proba.shape == labels.shape and np.isfinite(proba).all()
    assert ((0 <= proba) & (proba <= 1)).all()
    threshold, score = best_threshold(labels, proba)
    scores = fold_scores(labels, proba, splits, threshold)
    assert np.isclose(score, f1_score(labels, proba >= threshold, average='macro'))
    return dict(model=name, threshold=threshold, oof_macro_f1=score,
                fold_mean=float(scores.mean()), fold_std=float(scores.std()),
                cv_seconds=seconds)

best_params = {'C': 1.0, 'gamma': 'scale'}


Full training: 27309 | fold train sizes: [21847, 21847, 21847, 21847, 21848]


## From scratch: 5-fold full-training

Checkpoint OOF ditulis secara atomik setelah setiap fold selesai. Nilai NaN menandai fold yang belum selesai.

In [ ]:
scratch_models = []
oof_scratch = np.full(len(y), np.nan)
scratch_times = []
cv_warnings = []

def make_scratch():
    model = SVM(**BASE_PARAMS, **best_params)
    scratch_models.append(model)
    return model

with threadpool_limits(limits=1):
    for k, (tr, va) in enumerate(folds, 1):
        print(f'Mulai full fold {k}/{N_FOLDS}, train={len(tr)}, validasi={len(va)}', flush=True)
        started = time.perf_counter()
        with warnings.catch_warnings(record=True) as messages:
            warnings.simplefilter('always', RuntimeWarning)
            part = cross_val_proba(make_scratch, SVMPreprocessor, train, y, [(tr, va)])
        cv_warnings.extend(messages)
        oof_scratch[va] = part[va]
        assert np.isfinite(oof_scratch[va]).all()
        scratch_times.append(time.perf_counter() - started)
        temp_path = RUN_DIR / 'scratch_progress.tmp.npz'
        np.savez_compressed(temp_path, oof=oof_scratch, y=y, completed_folds=k,
                            fold_seconds=scratch_times,
                            iterations=[m.n_iter_ for m in scratch_models],
                            converged=[m.converged_ for m in scratch_models])
        temp_path.replace(RUN_DIR / 'scratch_progress.npz')
        print(f'Selesai fold {k}: {scratch_times[-1]:.1f}s, '
              f'iterasi={scratch_models[-1].n_iter_}, konvergen={scratch_models[-1].converged_}, '
              f'peringatan={len(messages)}', flush=True)

res_scratch = report('SVM from scratch', y, oof_scratch, folds, sum(scratch_times))
display(pd.DataFrame([res_scratch]).set_index('model').round(4))
display(pd.DataFrame([dict(fold=k, iterations=m.n_iter_, converged=m.converged_)
                      for k, m in enumerate(scratch_models, 1)]))
print('Peringatan termasuk kalibrasi:', len(cv_warnings))
for message in sorted({str(w.message) for w in cv_warnings}):
    print(message)


## Pembanding scikit-learn

Fold, preprocessing, dan parameter sama; kalibrasi sklearn memakai 5-fold internal dengan ensemble=False.

In [ ]:
class SklearnWrapper:
    def __init__(self, model):
        self.model = model

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]

reference_models = []

def make_reference():
    print(f'Mulai fold sklearn {len(reference_models) + 1}/{N_FOLDS}', flush=True)
    model = SklearnWrapper(CalibratedClassifierCV(
        SVC(kernel=KERNEL, **best_params, tol=BASE_PARAMS['tol'], random_state=SEED),
        method='sigmoid', ensemble=False,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)))
    reference_models.append(model)
    return model

started = time.perf_counter()
with threadpool_limits(limits=1):
    oof_sklearn = cross_val_proba(make_reference, SVMPreprocessor, train, y, folds)
res_sklearn = report('sklearn SVC', y, oof_sklearn, folds,
                     time.perf_counter() - started)
comparison = pd.DataFrame([res_scratch, res_sklearn]).set_index('model')
display(comparison.round(4))
per_fold = pd.DataFrame({
    'fold': np.arange(1, N_FOLDS + 1),
    'from_scratch': fold_scores(y, oof_scratch, folds, res_scratch['threshold']),
    'sklearn': fold_scores(y, oof_sklearn, folds, res_sklearn['threshold']),
})
display(per_fold.round(4))
pred_oof = (oof_scratch >= res_scratch['threshold']).astype(int)
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(confusion_matrix(y, pred_oof))
print('F1 kelas 0 dan 1:', f1_per_class(y, pred_oof).round(4))
print('Selisih OOF from scratch - sklearn:',
      round(res_scratch['oof_macro_f1'] - res_sklearn['oof_macro_f1'], 6))

## Diagnostik OOF full-training

In [ ]:
thresholds = np.round(np.arange(0.20, 0.81, 0.01), 2)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout='constrained')
for name, proba, result, color, style in [
    ('From scratch', oof_scratch, res_scratch, '#126782', '-'),
    ('scikit-learn', oof_sklearn, res_sklearn, '#b45309', '--'),
]:
    axes[0].plot(thresholds, [macro_f1(y, proba >= t) for t in thresholds],
                 label=name, color=color, linestyle=style)
    axes[0].scatter(result['threshold'], result['oof_macro_f1'], color=color, zorder=3)
axes[0].set(title='Macro F1 OOF terhadap threshold', xlabel='Threshold P(kelas 1)',
            ylabel='Macro F1')
axes[0].legend()
axes[1].plot(per_fold['fold'], per_fold['from_scratch'], 'o-',
             color='#126782', label='From scratch')
axes[1].plot(per_fold['fold'], per_fold['sklearn'], 's--',
             color='#b45309', label='scikit-learn')
axes[1].set(title='Macro F1 pada lima fold validasi', xlabel='Fold', ylabel='Macro F1',
            xticks=range(1, N_FOLDS + 1))
axes[1].legend()
for ax in axes:
    ax.grid(axis='y', alpha=0.2)
fig.suptitle(f'SVM {KERNEL}: {len(y):,} baris train bersih, full-training, seed {SEED}\n{best_params}',
             fontsize=12)
FIG_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_PATH, dpi=150)
plt.show()

In [ ]:
import json
np.savez_compressed(RUN_DIR / 'oof_complete.npz', y=y, scratch=oof_scratch, sklearn=oof_sklearn)
summary = dict(status='complete', parameters=dict(BASE_PARAMS, **best_params),
               train_rows=len(train), scratch=res_scratch, sklearn=res_sklearn,
               converged_folds=sum(bool(m.converged_) for m in scratch_models),
               warning_count=len(cv_warnings))
(RUN_DIR / 'results.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(comparison.round(4))
print('Run full selesai. Hasil:', RUN_DIR, flush=True)
